<a href="https://colab.research.google.com/github/Amyerm/ClassFiles/blob/main/Sesion10_Data_Profiling_Entregable_271756.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo:** Ari Daniel Mendoza Enrriquez

**Matrícula:** 271756

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [1]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [2]:
print("Columnas originales:")
print(df_marketing.columns)

# Copia para no modificar el original
df_marketing_renombrado = df_marketing.copy()

# Conver nombres a minúsculas
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower()

# Cambiamos algunos nombres para que sean mas claros
df_marketing_renombrado = df_marketing_renombrado.rename(columns={
    'year_birth': 'year_of_birth',
    'kidhome': 'kids_at_home',
    'mntwines': 'amount_spent_on_wine'
})

print("\nColumnas renombradas:")
print(df_marketing_renombrado.columns)

Columnas originales:
Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='object')

Columnas renombradas:
Index(['id', 'year_of_birth', 'education', 'marital_status', 'income',
       'kids_at_home', 'teenhome', 'dt_customer', 'recency',
       'amount_spent_on_wine', 'mntfruits', 'mntmeatproducts',
       'mntfishproducts', 'mntsweetproducts', 'mntgoldprods',
       'numdealspurchases', 'numwebpurchases', 'numcatalogpurchases',
       'numstorepurchases', 'numwebvisitsmonth', 'acceptedcmp3',
       'acceptedcmp4', 'acceptedcmp5', 'acce

---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [3]:
# Contar valores nulos antes de convertir la columna
nulos_antes = df_netflix['date_added'].isnull().sum()

# Conver columna a tipo fecha
df_netflix['date_added'] = pd.to_datetime(
    df_netflix['date_added'],
    format='mixed'
)

# Contar valores nulos después de la conversión
nulos_despues = df_netflix['date_added'].isnull().sum()

# Mostrar tipo de dato y comparar los nulos
print("Tipo de dato después de la conversión:")
print(df_netflix[['date_added']].dtypes)

print("\nNulos antes de la conversión:", nulos_antes)
print("Nulos después de la conversión:", nulos_despues)

Tipo de dato después de la conversión:
date_added    datetime64[ns]
dtype: object

Nulos antes de la conversión: 10
Nulos después de la conversión: 10


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [4]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [5]:
# Contar duplicados exactos
duplicados_exactos = df_marketing_dup.duplicated().sum()

# Contar duplicados con solamente el ID
duplicados_por_id = df_marketing_dup.duplicated(subset='ID').sum()

print("Duplicados exactos:", duplicados_exactos)
print("Duplicados por ID:", duplicados_por_id)
print("¿Los dos conteos coinciden?", duplicados_exactos == duplicados_por_id)

# Eliminar las filas duplicadas
df_marketing_sin_duplicados = df_marketing_dup.drop_duplicates()

# Comprobar número final de filas
print("\nFilas antes de eliminar duplicados:", len(df_marketing_dup))
print("Filas después de eliminar duplicados:", len(df_marketing_sin_duplicados))


Duplicados exactos: 2
Duplicados por ID: 2
¿Los dos conteos coinciden? True

Filas antes de eliminar duplicados: 2242
Filas después de eliminar duplicados: 2240


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [6]:
# Contar valores faltantes de cada columna
nulos_por_columna = df_netflix.isnull().sum()

print("Valores faltantes por columna:")
print(nulos_por_columna)

# Identificar filas con al menos un valor faltante
filas_con_faltantes = df_netflix.isnull().any(axis=1)

# Filtrar únicamente esas filas
df_con_faltantes = df_netflix[filas_con_faltantes]

# Contar filas con uno o más valores faltantes
total_filas_con_faltantes = filas_con_faltantes.sum()

print("\nTotal de filas con al menos un valor faltante:")
print(total_filas_con_faltantes)

# Mostrar primeras filas encontradas
print("\nEjemplo de filas con valores faltantes:")
display(df_con_faltantes.head())


Valores faltantes por columna:
show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64

Total de filas con al menos un valor faltante:
2979

Ejemplo de filas con valores faltantes:


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
11,s12,TV Show,1983,NaN,"Robert Więckiewicz, Maciej Musiał, Michalina O...","Poland, United States",2018-11-30,2018,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Dramas","In this dark alt-history thriller, a naïve law..."
12,s13,TV Show,1994,Diego Enrique Osorno,NaN,Mexico,2019-05-17,2019,TV-MA,1 Season,"Crime TV Shows, Docuseries, International TV S...",Archival video and new interviews examine Mexi...
16,s17,TV Show,09-Feb,NaN,"Shahd El Yaseen, Shaila Sabt, Hala, Hanadi Al-...",NaN,2019-03-20,2018,TV-14,1 Season,"International TV Shows, TV Dramas","As a psychology professor faces Alzheimer's, h..."
19,s20,Movie,'89,NaN,"Lee Dixon, Ian Wright, Paul Merson",United Kingdom,2018-05-16,2017,TV-PG,87 min,Sports Movies,"Mixing old footage with interviews, this is th..."


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [7]:
# Obtener nulos y el total de filas
nulos = df_netflix.isnull().sum()
total_filas = len(df_netflix)

# Calcular el porcentaje de completitud por columna
completitud = (1 - nulos / total_filas) * 100

print("Porcentaje de completitud por columna:")
print(completitud.round(2))

# Identificar la columna con menor completitud
columna_menor = completitud.idxmin()
porcentaje_menor = completitud.min()

print(f"\nLa columna con menor completitud es '{columna_menor}' con {porcentaje_menor:.2f}%.")

Porcentaje de completitud por columna:
show_id         100.00
type            100.00
title           100.00
director         69.32
cast             90.78
country          93.49
date_added       99.87
release_year    100.00
rating           99.91
duration        100.00
listed_in       100.00
description     100.00
dtype: float64

La columna con menor completitud es 'director' con 69.32%.


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [8]:
# Contar cuántas veces aparece cada estado civil
conteo_estado_civil = df_marketing['Marital_Status'].value_counts()

print("Cantidad de registros por estado civil:")
print(conteo_estado_civil)

# Identificar valores anómalos
print("\nValores anómalos encontrados: Alone, Absurd y YOLO")

# Escribir la decisión y su justificación
print("\nDecisión: Reclasificaría 'Alone' como 'Single' porque tienen un significado similar. "
      "Eliminaría los registros 'Absurd' y 'YOLO' porque no representan estados civiles válidos "
      "y no es posible conocer su categoría correcta.")

Cantidad de registros por estado civil:
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

Valores anómalos encontrados: Alone, Absurd y YOLO

Decisión: Reclasificaría 'Alone' como 'Single' porque tienen un significado similar. Eliminaría los registros 'Absurd' y 'YOLO' porque no representan estados civiles válidos y no es posible conocer su categoría correcta.


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [9]:
# Verificar cuáles valores cumplen el patrón indicado
cumple_patron = df_netflix['show_id'].str.match(r'^s\d+$')

# Calcular el porcentaje de cumplimiento
porcentaje_cumplimiento = cumple_patron.mean() * 100

print("¿Todos los valores cumplen el patrón?", cumple_patron.all())
print(f"Porcentaje de cumplimiento: {porcentaje_cumplimiento:.2f}%")

¿Todos los valores cumplen el patrón? True
Porcentaje de cumplimiento: 100.00%


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [10]:
# Mostrar resumen estadístico de la columna Year_Birth
print("Resumen de Year_Birth:")
print(df_marketing[['Year_Birth']].describe())

# Obtener el año de nacimiento mínimo
anio_minimo = df_marketing['Year_Birth'].min()
print("\nAño de nacimiento mínimo:", anio_minimo)

# Filtrar los años de nacimiento demasiado antiguos
anios_antiguos = df_marketing[df_marketing['Year_Birth'] < 1940]

print("\nRegistros con años de nacimiento anteriores a 1940:")
display(anios_antiguos[['ID', 'Year_Birth']].sort_values('Year_Birth'))

# Escribir la conclusión
print("Conclusión: Los años 1893, 1899 y 1900 son valores poco razonables para este dataset, "
      "por lo que los consideraría errores de captura.")

Resumen de Year_Birth:
        Year_Birth
count  2240.000000
mean   1968.805804
std      11.984069
min    1893.000000
25%    1959.000000
50%    1970.000000
75%    1977.000000
max    1996.000000

Año de nacimiento mínimo: 1893

Registros con años de nacimiento anteriores a 1940:


,ID,Year_Birth
239,11004,1893
339,1150,1899
192,7829,1900


Conclusión: Los años 1893, 1899 y 1900 son valores poco razonables para este dataset, por lo que los consideraría errores de captura.


---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [11]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [12]:
# Revisar tipos de datos antes de la corrección
print("Tipos de datos antes de la corrección:")
print(df_practica.dtypes)

# Convertir Income a numérico
df_practica['Income'] = pd.to_numeric(
    df_practica['Income'],
    errors='coerce'
)

# Comprobar tipo después de la conversión
print("\nTipo de dato de Income después de la corrección:")
print(df_practica['Income'].dtype)

Tipos de datos antes de la corrección:
ID                      int64
Year_Birth              int64
Education              object
Marital_Status         object
Income                 object
Kidhome                 int64
Teenhome                int64
Dt_Customer            object
Recency                 int64
MntWines                int64
MntFruits               int64
MntMeatProducts         int64
MntFishProducts         int64
MntSweetProducts        int64
MntGoldProds            int64
NumDealsPurchases       int64
NumWebPurchases         int64
NumCatalogPurchases     int64
NumStorePurchases       int64
NumWebVisitsMonth       int64
AcceptedCmp3            int64
AcceptedCmp4            int64
AcceptedCmp5            int64
AcceptedCmp1            int64
AcceptedCmp2            int64
Complain                int64
Z_CostContact           int64
Z_Revenue               int64
Response                int64
dtype: object

Tipo de dato de Income después de la corrección:
float64


In [13]:
# Contar filas duplicadas
duplicados = df_practica.duplicated().sum()
print("Filas duplicadas encontradas:", duplicados)

# Eliminar filas duplicadas
df_practica = df_practica.drop_duplicates().reset_index(drop=True)

# Verificar el resultado
print("Duplicados después de eliminarlos:", df_practica.duplicated().sum())
print("Número final de filas:", len(df_practica))

Filas duplicadas encontradas: 1
Duplicados después de eliminarlos: 0
Número final de filas: 15


In [14]:
# Contar los valores faltantes de cada columna
nulos_por_columna = df_practica.isnull().sum()

print("Valores faltantes por columna:")
print(nulos_por_columna)

Valores faltantes por columna:
ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 1
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Z_CostContact          0
Z_Revenue              0
Response               0
dtype: int64


In [15]:
# Revisar categorías existentes
categorias_estado_civil = df_practica['Marital_Status'].unique()

print("Categorías encontradas en Marital_Status:")
print(categorias_estado_civil)

print("\nConclusión: Las categorías tienen una escritura consistente, "
      "por lo que no necesitan normalización.")

Categorías encontradas en Marital_Status:
['Together' 'Single' 'Married' 'Divorced']

Conclusión: Las categorías tienen una escritura consistente, por lo que no necesitan normalización.


**Tu reporte de profiling:**

La columna Income tenía un tipo incorrecto por el valor “sesenta mil”, que fue inyectado para la práctica. Al convertirla a numérico, este valor se transformó en un dato faltante.
También se encontró y eliminó una fila duplicada inyectada, por lo que quedaron 15 registros.
En esta muestra no se encontraron otros valores faltantes provenientes de los datos reales.
Las categorías reales de Marital_Status tienen una escritura consistente, por lo que no requieren normalización.